# ST embeddings

adapted from: https://github.com/uhlerlab/spatialfusion/blob/main/workflows/unimodal-embeddings/scripts/unimodal-embeddings.py

In [1]:
import argparse
import logging
import os
import pathlib as pl
import sys
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import tifffile
import timm
import torch
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/scgpt_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys
sys.path.append('/insomnia001/depts/morpheus/users/me2982/models/scgpt/zero-shot-scfoundation/')

from sc_foundation_evals import scgpt_forward, data
from sc_foundation_evals.helpers.custom_logging import log

log.setLevel(logging.INFO)

/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/scgpt_env/lib/python3.10/site-packages/scgpt/model/model.py:21: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")
/insomnia001/depts/morpheus/users/me2982/miniconda3/envs/scgpt_env/lib/python3.10/site-packages/scgpt/model/multiomic_model.py:19: UserWarning: flash_attn is not installed
  warnings.warn("flash_attn is not installed")


In [3]:
def run_embed_scGPT(
    dataset_path: str,
    model_dir: str,  # path to the pre-trained model, 3 files are expected: model_weights (best_model.pt), model args (args.json), and model vocab (vocab.json)
    output_dir: str,  # output_dir is the path to which the results should be saved
    n_hvg: int,
    gene_col: str = "index",  # in which column in adata.obs are gene names stored? if they are in index, the index will be copied to a column with this name
    layer_key: str = "X",  # where the raw counts are stored?
    log_norm: bool = False,  # are the values log_norm already?
    seed: int = 42,
    max_seq_len: int = 1200,  # maximum sequence of the input is controlled by max_seq_len, here I'm using the pretrained default
    batch_size: int = 32,  # batch_size depends on available GPU memory; should be a multiple of 8
    input_bins: int = 51,
    model_run: str = "pretrained",
    num_workers: int = 0,  # if you can use multithreading specify num_workers
    output_name: str = "scGPT.parquet",
    scfoundation_dir: str | None = None,
) -> None:
    if scfoundation_dir:
        sys.path.append(scfoundation_dir)
    else:
        sys.path.append(str(DEFAULT_SCFOUNDATION_DIR))

    from sc_foundation_evals import scgpt_forward, data
    from sc_foundation_evals.helpers.custom_logging import log

    log.setLevel(logging.INFO)


    # create the model
    scgpt_model = scgpt_forward.scGPT_instance(
        saved_model_path=model_dir,
        model_run=model_run,
        batch_size=batch_size,
        save_dir=output_dir,
        num_workers=num_workers,
        explicit_save_dir=True,
    )

    # create config
    scgpt_model.create_configs(seed=seed, max_seq_len=max_seq_len, n_bins=input_bins)

    scgpt_model.load_pretrained_model()

    # This is to keep the model running if the amount of genes is small
    input_data = data.InputData(adata_dataset_path=dataset_path)
    vocab_list = scgpt_model.vocab.get_stoi().keys()

    adata = input_data.adata
    genes_in_vocab = adata.var_names.intersection(vocab_list)
    if len(genes_in_vocab) / len(adata.var_names) < 0.5:
        log.warning("Fewer than 50% of genes are found in the model vocab — continuing anyway.")
    
    adata._inplace_subset_var(genes_in_vocab)
    input_data.adata = adata

    input_data.preprocess_data(
        gene_vocab=vocab_list,
        model_type="scGPT",
        gene_col=gene_col,
        data_is_raw=not log_norm,
        counts_layer=layer_key,
        n_bins=input_bins,
        n_hvg=n_hvg,
    )

    scgpt_model.tokenize_data(
        data=input_data, input_layer_key="X_binned", include_zero_genes=False
    )

    scgpt_model.extract_embeddings(data=input_data)

    pd.DataFrame(
        input_data.adata.obsm["X_scGPT"],
        index=input_data.adata.obs["cell_id"] if "cell_id" in input_data.adata.obs.columns else input_data.adata.obs.index
    ).to_parquet(pl.Path(output_dir) / output_name)


dataset_path: str,
model_dir: str,  # path to the pre-trained model, 3 files are expected: model_weights (best_model.pt), model args (args.json), and model vocab (vocab.json)
output_dir: str,  # output_dir is the path to which the results should be saved
n_hvg: int,
gene_col: str = "index",  # in which column in adata.obs are gene names stored? if they are in index, the index will be copied to a column with this name
layer_key: str = "X",  # where the raw counts are stored?
log_norm: bool = False,  # are the values log_norm already?
seed: int = 42,
max_seq_len: int = 1200,  # maximum sequence of the input is controlled by max_seq_len, here I'm using the pretrained default
batch_size: int = 32,  # batch_size depends on available GPU memory; should be a multiple of 8
input_bins: int = 51,
model_run: str = "pretrained",
num_workers: int = 0,  # if you can use multithreading specify num_workers
output_name: str = "scGPT.parquet",
scfoundation_dir: str | None = None,

In [4]:
run_embed_scGPT(
    dataset_path = "/insomnia001/depts/morpheus/users/me2982/data/xenium_ovca_full/ovca_train.h5ad",
    model_dir = "/insomnia001/depts/morpheus/users/me2982/models/scgpt",
    output_dir = "/insomnia001/depts/morpheus/users/me2982/data/xenium_ovca_full/embeddings",
    n_hvg=1200,
    gene_col="index",
    layer_key="X",
    log_norm=False,
    seed=42,
    max_seq_len=1200,
    batch_size=16,
    input_bins=51,
    model_run="pretrained",
    num_workers=0,
    output_name="scGPT.parquet",
    scfoundation_dir="/insomnia001/depts/morpheus/users/me2982/models/scgpt/zero-shot-scfoundation/sc_foundation_evals"
)

INFO     | 2026-06-25 10:24:55 | Using device cuda
WARNING  | 2026-06-25 10:24:55 | Overriding pre-trained config['save_dir'] with /insomnia001/depts/morpheus/users/me2982/data/xenium_ovca_full/embeddings (was /scratch/ssd004/datasets/cellxgene/save/cellxgene_census_human-May23-08-36-2023)
WARNING  | 2026-06-25 10:24:55 | Overriding pre-trained config['max_seq_len'] with 1200 (was 1200)
INFO     | 2026-06-25 10:24:55 | Loading vocab from /insomnia001/depts/morpheus/users/me2982/models/scgpt/vocab.json


INFO     | 2026-06-25 10:24:57 | Loading model from /insomnia001/depts/morpheus/users/me2982/models/scgpt/best_model.pt
WARNING  | 2026-06-25 10:24:57 | Loading partial model params from /insomnia001/depts/morpheus/users/me2982/models/scgpt/best_model.pt
WARNING  | 2026-06-25 10:24:57 | Cannot load transformer_encoder.layers.0.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2026-06-25 10:24:57 | Cannot load transformer_encoder.layers.0.self_attn.in_proj_bias with shape torch.Size([1536])
WARNING  | 2026-06-25 10:24:57 | Cannot load transformer_encoder.layers.1.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2026-06-25 10:24:57 | Cannot load transformer_encoder.layers.1.self_attn.in_proj_bias with shape torch.Size([1536])
WARNING  | 2026-06-25 10:24:57 | Cannot load transformer_encoder.layers.2.self_attn.in_proj_weight with shape torch.Size([1536, 512])
WARNING  | 2026-06-25 10:24:57 | Cannot load transformer_encoder.layers.2.self_attn.in_pr

scGPT - INFO - Filtering genes by counts ...
scGPT - INFO - Normalizing total counts ...
scGPT - INFO - Log1p transforming ...
scGPT - INFO - Subsetting highly variable genes ...
scGPT - WARNING - No batch_key is provided, will use all cells for HVG selection.
scGPT - INFO - Binning data ...
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all zero rows. Please make sure this is expected. You can use the `filter_cell_by_counts` arg to filter out all zero rows.
scGPT - WARNING - The input data contains all z

INFO     | 2026-06-25 10:26:29 | Tokenizing data
INFO     | 2026-06-25 10:26:56 | Preparing dataloader
INFO     | 2026-06-25 10:26:56 | Saving config to /insomnia001/depts/morpheus/users/me2982/data/xenium_ovca_full/embeddings
INFO     | 2026-06-25 10:26:56 | Extracting embeddings
INFO     | 2026-06-25 10:26:57 | Extracting embeddings for batch 1/22528
INFO     | 2026-06-25 10:27:39 | Extracting embeddings for batch 2253/22528
INFO     | 2026-06-25 10:28:17 | Extracting embeddings for batch 4505/22528
INFO     | 2026-06-25 10:28:54 | Extracting embeddings for batch 6757/22528
INFO     | 2026-06-25 10:29:20 | Extracting embeddings for batch 9009/22528
INFO     | 2026-06-25 10:29:45 | Extracting embeddings for batch 11261/22528
INFO     | 2026-06-25 10:30:11 | Extracting embeddings for batch 13513/22528
INFO     | 2026-06-25 10:30:36 | Extracting embeddings for batch 15765/22528
INFO     | 2026-06-25 10:31:01 | Extracting embeddings for batch 18017/22528
INFO     | 2026-06-25 10:31:27 | 